# Importar Bibliotecas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os
from pathlib import Path
from typing import List, Dict, Any
from tqdm import tqdm # Para barra de progresso visual
from datetime import datetime

# Configurações e Constantes


In [ ]:
# Definir constantes para facilitar a manutenção e leitura
BASE_URL = 'https://investidor10.com.br/reits/'

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'

# Aumentado o timeout para requisições
REQUEST_TIMEOUT = 15 

# Aumentado o tempo de pausa para ser mais gentil com o servidor
SLEEP_TIME_SECONDS = 2 

# Número máximo de tentativas para uma requisição falha
MAX_RETRIES = 3 

# Usar pathlib para manipulação de caminhos de forma mais moderna e OS-agnóstica
OUTPUT_DIR = Path(r'C:\Users\RODRIGO\OneDrive\Documentos\Investimentos')

OUTPUT_FILENAME = 'Reits_indicadores_fundamentalistas.xlsx'

OUTPUT_DIVIDENDS_FILENAME = 'Rendimentos.xlsx'

# Lista de REITs para Análise

In [ ]:
REITS_TICKERS = [
    'AAT','ADC','AKR','AMH','AMT','APLE','ARE','AVB',
    'BRX','BXP',
    'CPT','CTRE','CUBE',
    'DEI','DLR',
    'EGP','ELS','EPR','EPRT','EQR','ESRT','ESS','EXR',
    'FR','FRT',
    'GLPI','GTY',
    'HASI','HIW',
    'IIPR','IRM','INVH',
    'KIM','KRC',
    'LAMR','LTC','LXP',
    'MAA',
    'NHI','NNN',
    'O','OHI','OLP',
    'PLD','PSA',
    'REG','REXR','RYN',
    'SBRA','SKT','SPG','STAG','STWD','SUI',
    'TRNO',
    'UDR','UHT',
    'VICI','VTR',
    'WELL','WY'
]

# Configuração dos indicadores desejados, incluindo função de limpeza para cada um

In [ ]:
# Isso garante que os dados sejam numéricos e no formato correto para análise.
INDICATORS_CONFIG = {
    'P/L': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'P/RECEITA (PSR)': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'P/VP': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'DIVIDEND YIELD (DY)': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'ROA': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'ROE': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'ROIC': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'MARGEM LÍQUIDA': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'MARGEM BRUTA': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'MARGEM OPERACIONAL': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'P/EBITDA': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'P/EBIT': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'P/ATIVO': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'VPA': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'LPA': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.') if str(x) != 'N/A' else 'nan')},
    'PATRIMÔNIO / ATIVOS': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'CAGR RECEITAS 5 ANOS': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100},
    'CAGR LUCROS 5 ANOS': {'clean_func': lambda x: float(str(x).replace('.', '').replace(',', '.').replace('%', '') if str(x) != 'N/A' else 'nan') / 100}
}
DESIRED_INDICATOR_NAMES = list(INDICATORS_CONFIG.keys())

# Funções Auxiliares

In [ ]:
def fetch_reit_page(reit_ticker: str, retries: int = MAX_RETRIES) -> requests.Response | None:
    """
    Tenta buscar a página web de um REIT, com retentativas em caso de falha.
    Retorna o objeto Response ou None em caso de erro persistente.
    """
    url = f'{BASE_URL}{reit_ticker.lower()}/'
    headers = {'User-Agent': USER_AGENT}

    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()  # Lança HTTPError para status 4xx/5xx
            return response
        except requests.exceptions.HTTPError as e:
            print(f"  -> Erro HTTP ({e.response.status_code}) para {reit_ticker} (Tentativa {attempt + 1}/{retries}).")
            if attempt < retries - 1:
                time.sleep(SLEEP_TIME_SECONDS * (attempt + 1)) # Backoff exponencial simples
            else:
                print(f"  -> Falha final para {reit_ticker} após {retries} tentativas.")
                return None
        except requests.exceptions.RequestException as e:
            print(f"  -> Erro de conexão para {reit_ticker} (Tentativa {attempt + 1}/{retries}): {e}.")
            if attempt < retries - 1:
                time.sleep(SLEEP_TIME_SECONDS * (attempt + 1))
            else:
                print(f"  -> Falha final para {reit_ticker} após {retries} tentativas.")
                return None
        except Exception as e:
            print(f"  -> Erro inesperado para {reit_ticker} (Tentativa {attempt + 1}/{retries}): {e}.")
            if attempt < retries - 1:
                time.sleep(SLEEP_TIME_SECONDS * (attempt + 1))
            else:
                print(f"  -> Falha final para {reit_ticker} após {retries} tentativas.")
                return None
    return None # Deveria ser inalcançável se retries > 0 e loop terminar

def parse_reit_indicators(html_content: str) -> Dict[str, str]:
    """
    Analisa o conteúdo HTML para extrair os indicadores fundamentalistas.
    Retorna um dicionário com os nomes dos indicadores e seus valores brutos.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    table_indicators_div = soup.find('div', id='table-indicators')
    
    indicators_on_page = {}

    if table_indicators_div:
        indicator_cells = table_indicators_div.find_all('div', class_='cell')
        for cell in indicator_cells:
            name_elem = cell.find('h3')
            value_elem = cell.find('div', class_='value').find('span')
            if name_elem and value_elem:
                name = name_elem.get_text(strip=True)
                value = value_elem.get_text(strip=True)
                indicators_on_page[name] = value
    return indicators_on_page

def clean_indicator_value(indicator_name: str, raw_value: str) -> Any:
    """
    Limpa e converte o valor de um indicador para o tipo numérico apropriado,
    usando a função de limpeza definida em INDICATORS_CONFIG.
    """
    config = INDICATORS_CONFIG.get(indicator_name)
    if config and 'clean_func' in config:
        try:
            return config['clean_func'](raw_value)
        except (ValueError, AttributeError):
            # Retorna None (que pandas trata como NaN) se a limpeza falhar
            return None
    return raw_value # Retorna o valor bruto se não houver função de limpeza definida

def scrape_dividends(html_content: str, reit_ticker: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(html_content, 'html.parser')
    reit_dividends: List[Dict[str, Any]] = []
    
    # Tenta encontrar a tabela de proventos/dividendos
    # O Investidor 10 costuma colocar dentro de uma div com id 'proventos_tab' ou similar
    table = soup.find('table', {'id': 'table-dividends-history'})
    
    if not table:
        # Busca alternativa: qualquer tabela que tenha a palavra "Data COM" ou "Pagamento"
        tables = soup.find_all('table')
        for t in tables:
            if "Pagamento" in t.get_text():
                table = t
                break
    
    if not table:
        return reit_dividends

    current_year = datetime.now().year
    years_to_scrape = range(current_year - 5, current_year + 1)

    rows = table.find_all('tr')[1:] # Pula o cabeçalho
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 4:
            # No Investidor 10 a ordem costuma ser: Tipo, Data COM, Pagamento, Valor
            date_str = cols[2].get_text(strip=True) 
            value_str = cols[3].get_text(strip=True)

            try:
                if date_str and date_str != "-":
                    payment_date = datetime.strptime(date_str, '%d/%m/%Y')
                    if payment_date.year in years_to_scrape:
                        # Limpeza robusta do valor
                        val = value_str.replace('R$', '').replace('$', '').replace('.', '').replace(',', '.').strip()
                        cleaned_value = float(val)
                        
                        reit_dividends.append({
                            'Reit': reit_ticker,
                            'Data Pagamento': payment_date.strftime('%Y-%m-%d'),
                            'Dividendo por Ação': cleaned_value
                        })
            except Exception:
                continue

    return reit_dividends

# Função Principal

In [ ]:
def main():
    """
    Função principal que orquestra o processo de web scraping,
    coleta, limpeza e salvamento dos dados dos REITs.
    """
    print("Iniciando o web scraping de indicadores fundamentalistas e dividendos de REITs...")

    all_reits_indicators_data: List[Dict[str, Any]] = []
    all_reits_dividends_data: List[Dict[str, Any]] = []

    # Usar tqdm para uma barra de progresso visual, tornando a experiência mais intuitiva
    for reit_ticker in tqdm(REITS_TICKERS, desc="Coletando dados de REITs"):
        reit_indicator_data: Dict[str, Any] = {'Reit': reit_ticker}
        
        response = fetch_reit_page(reit_ticker)
        
        if response:
            # --- Coleta de Indicadores ---
            indicators_found = parse_reit_indicators(response.text)
            for indicator_name in DESIRED_INDICATOR_NAMES:
                raw_value = indicators_found.get(indicator_name, 'N/A')
                cleaned_value = clean_indicator_value(indicator_name, raw_value)
                reit_indicator_data[indicator_name] = cleaned_value
            reit_indicator_data['Status_Coleta'] = 'Sucesso'

            # --- Coleta de Dividendos ---
            dividends_for_reit = scrape_dividends(response.text, reit_ticker)
            all_reits_dividends_data.extend(dividends_for_reit)

        else:
            # Se a busca da página falhou, preenche com None e marca o status para indicadores
            for indicator_name in DESIRED_INDICATOR_NAMES:
                reit_indicator_data[indicator_name] = None
            reit_indicator_data['Status_Coleta'] = 'Falha na Requisição/Conexão'
            # Dividendos para este REIT já estarão vazios se a resposta for None

        all_reits_indicators_data.append(reit_indicator_data)
        
        time.sleep(SLEEP_TIME_SECONDS) # Pausa para não sobrecarregar o servidor

    # --- Processamento e Exportação de Indicadores Fundamentalistas ---
    if all_reits_indicators_data:
        df_indicators = pd.DataFrame(all_reits_indicators_data)
        
        # Garante que todas as colunas desejadas estejam presentes, preenchendo com None/NaN
        for col in DESIRED_INDICATOR_NAMES:
            if col not in df_indicators.columns:
                df_indicators[col] = None
        
        # Reordena as colunas para uma apresentação consistente
        final_columns_order_indicators = ['Reit'] + DESIRED_INDICATOR_NAMES + ['Status_Coleta']
        df_indicators = df_indicators[final_columns_order_indicators]
        
        # Cria o diretório de saída se ele não existir
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_excel_path_indicators = OUTPUT_DIR / OUTPUT_FILENAME

        try:
            df_indicators.to_excel(output_excel_path_indicators, index=False)
            print(f'\nWeb scraping de indicadores concluído! Os dados foram salvos em "{output_excel_path_indicators}".')
        except Exception as e:
            print(f'\nErro ao salvar o arquivo Excel de indicadores: {e}')
    else:
        print('\nNenhum dado de indicadores foi coletado. Verifique os tickers e a estrutura do site.')

    # --- Processamento e Exportação de Dividendos ---
# --- Processamento e Exportação de Dividendos ---
    if all_reits_dividends_data:
        df_dividends = pd.DataFrame(all_reits_dividends_data)
        
        # Caminho completo para salvar
        output_excel_path_dividends = OUTPUT_DIR / OUTPUT_DIVIDENDS_FILENAME
        
        try:
            # SALVAR DIRETAMENTE NO EXCEL (Igual você fez com os indicadores)
            df_dividends.to_excel(output_excel_path_dividends, index=False)
            print(f'\nArquivo de dividendos salvo com sucesso em: "{output_excel_path_dividends}"')
        except Exception as e:
            print(f'\nErro ao salvar o arquivo Excel de dividendos: {e}')
    else:
        print('\nNenhum dado de dividendos foi coletado. O site pode estar bloqueando ou a tabela mudou.')

# Não esqueça de chamar a função main no final do arquivo
if __name__ == "__main__":
    main()